In [66]:
#-- Import libraries. --#
import os
import json
import folium
import pandas as pd
from folium.plugins import Search

In [88]:
#-- Read data. --#
df_viz = pd.read_csv('../Processed/efficiency_stations.csv')

In [89]:
df_viz['road_name'] = df_viz['declared_r'].fillna('Unknown Road')
invalid_names = ['Unknown Road', 'UNKNOWN', '', 'nan', 'None']
df_viz = df_viz[~df_viz['road_name'].isin(invalid_names)]
df_viz = df_viz.dropna(subset=['road_name'])

In [90]:
if 'vs_average_pct' not in df_viz.columns:
    df_viz['vs_average_pct'] = ((df_viz['efficiency_index'] - 1) * 100).round(1)

In [91]:
m = folium.Map(location=[-37.815, 145.07], zoom_start=14, tiles=None)

In [92]:
folium.TileLayer('cartodbpositron', name='Light Mode').add_to(m)
folium.TileLayer('cartodbdark_matter', name='Dark Mode').add_to(m)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}', 
    attr='Google', name='Satellite View'
).add_to(m)

In [96]:
stations_js = df_viz[['x', 'y', 'tfm_id', 'road_name', 'efficiency_index', 'vs_average_pct', 'aadt_allve']].to_dict('records')

features = []
for s in stations_js:
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [s['x'], s['y']]},
        "properties": {
            "search_key": f"Stations {s['tfm_id']} - {s['road_name']}",
            "road": s['road_name'],
            "eff": f"{s['efficiency_index']}x",
            "pct": f"{s['vs_average_pct']}%",
            "aadt": f"{int(s['aadt_allve']):,}"
        }
    })

station_layer = folium.GeoJson(
    {"type": "FeatureCollection", "features": features},
    name="Efficiency Stations",
    marker=folium.CircleMarker(radius=6, color='#00FFFF', weight=1, fill=True, fill_color='#6A0DAD'),
    popup=folium.GeoJsonPopup(
        fields=['road', 'eff', 'pct', 'aadt'], 
        aliases=['📍 Path:', '📈 Efifciency:', '📊 Average:', '🚗 Traffic Flow:']
    )
).add_to(m)

In [97]:
search_control = Search(
    layer=station_layer,
    geom_type='Point',
    placeholder='Tìm trạm hoặc tên đường...',
    collapsed=False,
    search_label='search_key'
).add_to(m)

In [95]:
js_code = f"""
<script>
var stations = {json.dumps(stations_js)}; 
var currentHighlight = null;

function clearHighlight() {{ if (currentHighlight) {{ m.removeLayer(currentHighlight); }} }}

// Lắng nghe sự kiện từ ô search duy nhất của Folium
m.on('search:locationfound', function(e) {{
    clearHighlight();
    var searchText = e.text.toLowerCase();
    
    // Logic highlight vòng tròn nhỏ (radius 15)
    currentHighlight = L.circle([e.latlng.lat, e.latlng.lng], 
                       {{radius: 15, color: '#FFBF00', weight: 4, fillOpacity: 0.3}}).addTo(m);
    
    // Nếu là search theo đường, nối line Royal Blue
    var roadName = searchText.includes(' - ') ? searchText.split(' - ')[1] : searchText;
    var roadPoints = stations.filter(s => s.road_name.toLowerCase() === roadName).map(s => [s.y, s.x]);
    
    if (roadPoints.length > 1) {{
        roadPoints.sort((a, b) => a[0] - b[0]); 
        var line = L.polyline(roadPoints, {{color: '#4169E1', weight: 8, opacity: 0.8}}).addTo(m);
        m.fitBounds(line.getBounds());
    }}
}});
</script>
"""
m.get_root().html.add_child(folium.Element(js_code))

# 8. Sidebar: Bảng Station List (Chỉ hiện trạm có tên)
table_html = f"""
<div style="position: fixed; bottom: 20px; left: 10px; width: 340px; height: 300px; 
            z-index:9999; background: white; padding: 12px; border-radius: 10px; 
            border: 2px solid #6A0DAD; overflow-y: auto; font-family: sans-serif; box-shadow: 0 4px 10px rgba(0,0,0,0.2);">
    <h4 style="margin-top:0; color:#6A0DAD; border-bottom: 2px solid #6A0DAD; padding-bottom:5px;">📍 Quality Station List</h4>
    <table style="width:100%; font-size: 11px; border-collapse: collapse; text-align: left;">
        <thead>
            <tr style="background:#6A0DAD; color:white;">
                <th style="padding:5px;">ID</th><th style="padding:5px;">Road Name</th><th style="padding:5px;">Eff</th>
            </tr>
        </thead>
        <tbody>
"""
for _, row in df_viz.sort_values('efficiency_index', ascending=False).head(20).iterrows():
    table_html += f"""
        <tr onclick="m.flyTo([{row['y']}, {row['x']}], 18)" style="cursor:pointer; border-bottom:1px solid #eee;">
            <td style="padding:5px;">{row['tfm_id']}</td>
            <td style="padding:5px;">{row['road_name']}</td>
            <td style="padding:5px; font-weight:bold; color:#4169E1;">{row['efficiency_index']}x</td>
        </tr>"""
table_html += "</tbody></table></div>"
m.get_root().html.add_child(folium.Element(table_html))

In [98]:
folium.LayerControl(position='topright', collapsed=False).add_to(m)
m.save('Boroondara_Efficiency_Final.html')
print("✅ Finsihed")

✅ Finsihed
